In [ ]:
# import requests
# import json

# with open('config.json') as f:
#     config = json.load(f)

# api_key = config['apiKey']

# url = 'http://apis.data.go.kr/B552584/EvCharger/getChargerInfo'
# params ={'serviceKey' : api_key, 'pageNo' : '1', 'numOfRows' : '10', 'zcode' : '11', 'datatype' : 'JSON' }

# response = requests.get(url, params=params)
# print(response.content)

b'<?xml version="1.0" encoding="UTF-8" standalone="yes"?><response><header><resultCode>00</resultCode><resultMsg>NORMAL SERVICE.</resultMsg><totalCount>63690</totalCount><pageNo>1</pageNo><numOfRows>10</numOfRows></header><body><items><item><statNm>\xeb\x82\x99\xec\x84\xb1\xeb\x8c\x80\xeb\x8f\x99\xec\xa3\xbc\xeb\xaf\xbc\xec\x84\xbc\xed\x84\xb0</statNm><statId>ME174013</statId><chgerId>01</chgerId><chgerType>06</chgerType><addr>\xec\x84\x9c\xec\x9a\xb8\xed\x8a\xb9\xeb\xb3\x84\xec\x8b\x9c \xea\xb4\x80\xec\x95\x85\xea\xb5\xac \xeb\x82\x99\xec\x84\xb1\xeb\x8c\x80\xeb\xa1\x9c4\xea\xb0\x80\xea\xb8\xb8 5</addr><addrDetail>null</addrDetail><location>null</location><lat>37.476296</lat><lng>126.9583876</lng><useTime>24\xec\x8b\x9c\xea\xb0\x84 \xec\x9d\xb4\xec\x9a\xa9\xea\xb0\x80\xeb\x8a\xa5</useTime><busiId>ME</busiId><bnm>\xed\x99\x98\xea\xb2\xbd\xeb\xb6\x80</bnm><busiNm>\xed\x99\x98\xea\xb2\xbd\xeb\xb6\x80</busiNm><busiCall>1661-9408</busiCall><stat>2</stat><statUpdDt>20250514163610</statUpdDt

In [ ]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import json
import time  # 호출 지연용
import os

with open('config.json') as f:
    config = json.load(f)

api_key = config['apiKey']

# API 기본 정보
url = 'http://apis.data.go.kr/B552584/EvCharger/getChargerInfo' 

# 요청 파라미터
params = {
    'ServiceKey': api_key,
    'pageNo': '1',
    'numOfRows': '9999',
    'zcode': '11',  # 서울특별시 예시
    'period': '5'   # 상태 갱신 범위 (단위: 분, 선택사항)
}

# API 호출
response = requests.get(url, params=params)

# XML 파싱
root = ET.fromstring(response.content)

# item 태그 반복
items = []
for item in root.iter('item'):
    item_data = {}
    for child in item:
        item_data[child.tag] = child.text
    items.append(item_data)

# pandas DataFrame으로 변환
df = pd.DataFrame(items)
print(df.head())


        statNm    statId chgerId chgerType                  addr addrDetail  \
0     낙성대동주민센터  ME174013      01        06   서울특별시 관악구 낙성대로4가길 5       null   
1     롯데마트 중계점  ME174018      01        06     서울특별시 노원구 노원로 330    옥상 매장 앞   
2       서울추모공원  ME174027      01        06  서울특별시 서초구 양재대로12길 74      1층 입구   
3  아시아공원 공영주차장  ME174028      01        06      서울특별시 송파구 잠실동 84       우측 끝   
4  아시아공원 공영주차장  ME174028      02        06      서울특별시 송파구 잠실동 84       우측 끝   

  location         lat          lng    useTime  ... kind kindDetail  \
0     null   37.476296  126.9583876  24시간 이용가능  ...   G0       G003   
1     null  37.6466673  127.0715514  24시간 이용가능  ...   E0       E001   
2     null  37.4536062  127.0428005  24시간 이용가능  ...   A0       A004   
3     null  37.5102969  127.0788121  24시간 이용가능  ...   B0       B001   
4     null  37.5102969  127.0788121  24시간 이용가능  ...   B0       B001   

  parkingFree  note limitYn limitDetail delYn delDetail trafficYn  year  
0           Y  None     

In [ ]:
# import requests
# import xml.etree.ElementTree as ET
# import pandas as pd
# import time  # 호출 지연용
# import os

# 1. 기본 설정
url = 'http://apis.data.go.kr/B552584/EvCharger/getChargerInfo'
rows_per_page = 2000

# 2. 먼저 totalCount 가져오기
params = {
    'serviceKey': api_key,
    'pageNo': 1,
    'numOfRows': 1,
    'zcode': '36',         # 서울시 코드 (zcode)
    'dataType': 'XML'
}

response = requests.get(url, params=params)
root = ET.fromstring(response.content)
total_count = int(root.find('.//totalCount').text)
total_pages = (total_count + rows_per_page - 1) // rows_per_page

print(f"총 개수: {total_count} / 총 페이지 수: {total_pages}")

# 폴더 생성 (부분 저장용)
os.makedirs("pages", exist_ok=True)

# 데이터 수집 및 페이지별 저장
all_data = []

for page in range(1, total_pages + 1):
    print(f"[{page}/{total_pages}] 페이지 수집 중...")
    
    params.update({
        'pageNo': page,
        'numOfRows': rows_per_page
    })
    
    try:
        response = requests.get(url, params=params, timeout=10)
        root = ET.fromstring(response.content)
        
        page_data = []
        for item in root.iter('item'):
            row = {child.tag: child.text for child in item}
            all_data.append(row)
            page_data.append(row)
        
        # 페이지별 CSV 저장
        df_page = pd.DataFrame(page_data)
        df_page.to_csv(f'pages/page_{page:02}.csv', index=False, encoding='euc-kr')
        print(f"  → page_{page:02}.csv 저장 완료")

        time.sleep(5)

    except Exception as e:
        print(f"⚠️ 페이지 {page} 실패: {e}")
        break

# 전체 데이터 최종 저장
df_all = pd.DataFrame(all_data)
df_all.to_csv('세종시_전기차충전소_전체.csv', index=False, encoding='euc-kr')
df_all.to_csv('세종시_전기차충전소_전체_utf인코딩.csv', index=False, encoding='utf-8-sig')
print("전체 CSV 저장 완료!")


총 개수: 5409 / 총 페이지 수: 3
[1/3] 페이지 수집 중...
  → page_01.csv 저장 완료
[2/3] 페이지 수집 중...
  → page_02.csv 저장 완료
[3/3] 페이지 수집 중...
  → page_03.csv 저장 완료
전체 CSV 저장 완료!


In [ ]:
df_all.to_csv('서울시_전기차충전소_전체_utf인코딩.csv', index=False, encoding='utf-8-sig')

In [4]:
df

,statNm,statId,chgerId,chgerType,addr,addrDetail,location,lat,lng,useTime,...,kind,kindDetail,parkingFree,note,limitYn,limitDetail,delYn,delDetail,trafficYn,year
0,낙성대동주민센터,ME174013,01,06,서울특별시 관악구 낙성대로4가길 5,null,null,37.476296,126.9583876,24시간 이용가능,...,G0,G003,Y,None,N,None,N,None,N,2017
1,롯데마트 중계점,ME174018,01,06,서울특별시 노원구 노원로 330,옥상 매장 앞,null,37.6466673,127.0715514,24시간 이용가능,...,E0,E001,N,None,N,None,N,None,N,2017
2,서울추모공원,ME174027,01,06,서울특별시 서초구 양재대로12길 74,1층 입구,null,37.4536062,127.0428005,24시간 이용가능,...,A0,A004,N,None,N,None,N,None,N,2017
3,아시아공원 공영주차장,ME174028,01,06,서울특별시 송파구 잠실동 84,우측 끝,null,37.5102969,127.0788121,24시간 이용가능,...,B0,B001,N,None,N,None,N,None,N,2017
4,아시아공원 공영주차장,ME174028,02,06,서울특별시 송파구 잠실동 84,우측 끝,null,37.5102969,127.0788121,24시간 이용가능,...,B0,B001,N,None,N,None,N,None,N,2017
5,아시아공원 공영주차장,ME174028,03,06,서울특별시 송파구 잠실동 84,우측 끝,null,37.5102969,127.0788121,24시간 이용가능,...,B0,B001,N,None,N,None,N,None,N,2017
6,롯데마트 송파점,ME174029,01,06,서울특별시 송파구 중대로 80,지하4층 C8,null,37.4918392,127.1178931,24시간 이용가능,...,E0,E001,N,None,N,None,N,None,N,2017
7,현대자동차 수색대리점,ME174037,01,06,서울특별시 은평구 수색로 342-1,null,null,37.5867665,126.8880299,24시간 이용가능,...,F0,F001,N,None,N,None,N,None,N,2017
8,태화빌딩,ME174039,01,06,서울특별시 종로구 인사동5길 29,지상주차장,null,37.5718148,126.9850443,24시간 이용가능,...,H0,H003,N,None,N,None,N,None,N,2017
9,서울만남(부산) 휴게소,ME178009,01,06,서울특별시 서초구 양재대로12길 73-71 (원지동),null,null,37.4600218,127.0420378,24시간 이용가능,...,C0,C001,Y,None,N,None,N,None,N,2017


In [7]:
df.to_csv('api_sample.csv', index=False, encoding='euc-kr')